# libs

In [ ]:
import math
import resource

import galsim
import holoviews
import numpy
import panel
from functions_cdfs import (
    binned_style,
    get_half_light_radius,
    get_inv_cdfs,
    get_magnitude,
    get_sersic_index,
)
from functions_filters import get_field_cosmos
from functions_images import GSPARAMS, build_full_image, build_psf, get_constants, get_offset
from functions_lhs import adding_cosmos_catalog_galaxies, draw_lhs, draw_lhs_gal, project_lhs_gal
from matplotlib import pyplot
from pandas import read_csv
from plots import comparaison_3d, diff_binned_continuous, lhs_visu_single_test, sky_coverage
from scipy import stats
from utils import DIRS, DTYPE_16, DTYPE_32, render_psf

holoviews.extension("bokeh", "plotly")
panel.extension()

resource.setrlimit(resource.RLIMIT_NOFILE, (50000, 50000))

PIXEL_SCALE, PATCH_SIZE, IMG_SIZE, EXPTIME, GAIN, readoutnoise, SKY_BKG, ZP = get_constants()
psf = build_psf("airy")

DIR_WIP = DIRS["pisco_euclid_plots"] / "wip"

# sersic

In [ ]:
sample = "23.5"
cosmos_cat = galsim.COSMOSCatalog(
    sample=sample, dir=DIRS["cosmos"], exclusion_level="none", exptime=1500
)

non_filtered_param = cosmos_cat.param_cat

sersic_cosmos = non_filtered_param["sersicfit"][:, 2]

In [ ]:
eps = 5e-2

possible_sersic = numpy.array(read_csv(DIRS["cosmos"] / "possible_sersic.csv")).squeeze()
diff = numpy.abs(sersic_cosmos[:, None] - possible_sersic[None, :])
mask = numpy.any(diff < eps, axis=1)

filtered_cosmos = sersic_cosmos[mask]
print(filtered_cosmos.shape[0])

_ = pyplot.hist(filtered_cosmos, bins=100, density=True)
_ = pyplot.hist(possible_sersic, bins=100, density=True, color="orange")

In [ ]:
NB_OF_GALAXIES = 20_000
PERCENTAGE_COSMOS = 20
version = "9999"
seed_database = 2222

lhs_gal = draw_lhs_gal(False, seed_database, NB_OF_GALAXIES)
lhs_gal = project_lhs_gal(hypercube=lhs_gal, ellips_style="gaussian", pixel_out=False)

nb_galaxies_cosmos = int(PERCENTAGE_COSMOS * NB_OF_GALAXIES / 100)
rng_local = numpy.random.default_rng(seed=seed_database)
indices = rng_local.choice(NB_OF_GALAXIES, nb_galaxies_cosmos, replace=False)
indices = numpy.random.choice(NB_OF_GALAXIES, nb_galaxies_cosmos, replace=False)
if PERCENTAGE_COSMOS > 0:
    lhs_gal = adding_cosmos_catalog_galaxies(lhs_gal, numpy.sort(indices), version)

In [ ]:
lhs_visu_single_test(DIR_WIP, lhs_gal[:, 4:6], alpha=0.5, bins=11, s=1)
lhs_visu_single_test(DIR_WIP, lhs_gal[:, [4, 6]], alpha=0.5, bins=11, s=1)

# check binned VS continuous


In [ ]:
NB_OF_GALAXIES = 20_000

list_mag, list_hlr, list_ns, list_e1, list_e2 = binned_style(NB_OF_GALAXIES)

hypercube = draw_lhs(2222, NB_OF_GALAXIES, dim=5)
inv_cdf_mag_fn, inv_cdfs_hlr_fn = get_inv_cdfs()
hypercube[:, 0], binned_mags = get_magnitude(hypercube[:, 0], inv_cdf_mag_fn)
hypercube[:, 1] = get_half_light_radius(hypercube[:, 1], binned_mags, inv_cdfs_hlr_fn)
hypercube[:, 2] = get_sersic_index(hypercube[:, 2], binned_mags)
hypercube[:, 3:5] = stats.norm.ppf(hypercube[:, 3:5], loc=0, scale=0.26)

In [ ]:
color = False
diff_binned_continuous(
    list_mag, list_hlr, list_ns, list_e1, list_e2, hypercube, DIR_WIP, color=color
)

# check replacing


In [ ]:
NB_OF_GALAXIES = 20_000
PERCENTAGE_COSMOS = 20
version = "9999"
seed_database = 2222

lhs_gal = draw_lhs_gal(False, seed_database, NB_OF_GALAXIES)
lhs_gal = project_lhs_gal(hypercube=lhs_gal, ellips_style="gaussian", pixel_out=False)

nb_galaxies_cosmos = int(PERCENTAGE_COSMOS * NB_OF_GALAXIES / 100)
rng_local = numpy.random.default_rng(seed=seed_database)
indices = rng_local.choice(NB_OF_GALAXIES, nb_galaxies_cosmos, replace=False)
indices = numpy.random.choice(NB_OF_GALAXIES, nb_galaxies_cosmos, replace=False)

check_columns = [0, 1, 5, 6]  # e1 e2 hlr sersic
catalog_targets = lhs_gal[:, check_columns][indices]

filtered_params = read_csv(DIRS["cosmos"] / "filtered_params.csv")
check_columns_cosmos = ["e1", "e2", "hlr", "sersic"]

catalog_features = filtered_params[check_columns_cosmos].values

results_features = {}
for SCALING in ["min_max", "z_score"]:
    for NORMA in ["euclidian", "mahalanobis"]:
        lhs_gal = adding_cosmos_catalog_galaxies(lhs_gal, indices, version, SCALING, NORMA)
        results_features[(SCALING, NORMA)] = lhs_gal[numpy.ix_(indices, check_columns)]
with_all_catalog = False
NB_CHOICES = 10
choices = numpy.random.choice(nb_galaxies_cosmos, size=NB_CHOICES, replace=False)
comparaison_3d(
    catalog_features, catalog_targets, results_features, choices, DIR_WIP, with_all_catalog
)

# mollmoll


In [ ]:
fields_23 = get_field_cosmos(sample="23.5")
fields_25 = get_field_cosmos(sample="25.2")
ra_23 = numpy.array(fields_23["RA"])
dec_23 = numpy.array(fields_23["DEC"])
ra_25 = numpy.array(fields_25["RA"])
dec_25 = numpy.array(fields_25["DEC"])

sky_coverage(ra_23, dec_23, ra_25, dec_25, DIRS["cosmos_plots"])

# f16 psf

In [ ]:
psf_1 = render_psf(psf, IMG_SIZE, DTYPE_16, PIXEL_SCALE)
psf_2 = render_psf(psf, IMG_SIZE, DTYPE_32, PIXEL_SCALE)
psf_3 = render_psf(psf, PATCH_SIZE, DTYPE_16, PIXEL_SCALE)
psf_4 = render_psf(psf, PATCH_SIZE, DTYPE_32, PIXEL_SCALE)

fig, axes = pyplot.subplots(2, 4, figsize=(20, 10))
psfs = [psf_1, psf_2, psf_3, psf_4]

for i in range(4):
    axes[0, i].imshow(psfs[i], origin="lower", cmap="viridis")
    axes[1, i].imshow(numpy.log(psfs[i]), origin="lower", cmap="viridis")

fig.tight_layout()
fig.savefig(DIR_WIP / "psf_16_32.pdf", dpi=300)
fig.show()

# f16 image

In [ ]:
NB_OF_GALAXIES = 10_000
lhs_gal_total = draw_lhs_gal(False, 222, NB_OF_GALAXIES)
lhs_gal_total = project_lhs_gal(hypercube=lhs_gal_total, ellips_style="gaussian", pixel_out=False)

In [ ]:
start = 100
lhs_gal = lhs_gal_total[start : start + 10]

full_image = build_full_image(IMG_SIZE, PIXEL_SCALE)
noise_image = build_full_image(IMG_SIZE, PIXEL_SCALE)

flux = EXPTIME / GAIN * 10 ** (-(lhs_gal[:, 4] - ZP) / 2.5)

for i in range(10):
    gal = galsim.Sersic(
        n=lhs_gal[i, 6],
        half_light_radius=lhs_gal[i, 5],
        flux=flux[i],
        gsparams=GSPARAMS,
        trunc=lhs_gal[i, 5] * 4.5,
    )
    gal.shear(e1=lhs_gal[i, 0], e2=lhs_gal[i, 1])

    final = galsim.Convolve([psf, gal])
    image_pos, ix_nominal, iy_nominal, offset = get_offset(
        lhs_gal[i, 2], lhs_gal[i, 3], lhs_gal[i, 7], lhs_gal[i, 8]
    )
    stamp = galsim.ImageF(PATCH_SIZE, PATCH_SIZE, scale=PIXEL_SCALE)
    final.drawImage(image=stamp, wcs=full_image.wcs.local(image_pos), offset=offset)
    stamp.setCenter(ix_nominal, iy_nominal)
    bounds = stamp.bounds & full_image.bounds
    full_image[bounds] += stamp[bounds]

F_SKY = PIXEL_SCALE ** (2) * EXPTIME * 10 ** (-(SKY_BKG - ZP) / 2.5)  # e-/pixel
variance = (numpy.sqrt(readoutnoise**2 + F_SKY) / GAIN) ** 2  # e- -> ADU
noise = galsim.GaussianNoise(rng=galsim.BaseDeviate(2222), sigma=math.sqrt(variance))
noise_image.addNoise(noise)
full_image += noise_image

image = full_image.array
print(image.min(), image.max())

In [ ]:
fig, axes = pyplot.subplots(2, 2, figsize=(10, 10))
cmap = "viridis"

images = [
    numpy.float32(image),
    numpy.float32(numpy.arcsinh(image)),
    numpy.float16(image),
    numpy.float16(numpy.arcsinh(image)),
]

for ax, img in zip(axes.flat, images):
    im = ax.imshow(img, origin="lower", cmap="viridis")
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
fig.tight_layout()
fig.show()